# Agent for S05E04: Production-Ready Agent

This notebook implements a production-ready agent with features discussed in the S05E04 lesson, including error handling, logging, observability, and scalable architecture.

## Features:
1. Structured logging
2. Error handling and retries
3. Performance monitoring
4. Configurable settings
5. Human-in-the-loop capabilities

In [ ]:
# Install required packages
!pip install openai loguru tenacity

In [ ]:
from openai import OpenAI
from loguru import logger
from tenacity import retry, stop_after_attempt, wait_exponential
import time
import json

class ProductionAgent:
    def __init__(self, api_key, config=None):
        self.client = OpenAI(api_key=api_key)
        self.config = config or {
            'max_retries': 3,
            'timeout': 30,
            'log_level': 'INFO'
        }
        self.setup_logging()
        self.metrics = {
            'requests': 0,
            'errors': 0,
            'avg_response_time': 0
        }
        logger.info("Production Agent initialized")
    
    def setup_logging(self):
        """Setup structured logging"""
        logger.remove()  # Remove default handler
        logger.add(
            "agent.log",
            format="{time} | {level} | {message} | {extra}",
            level=self.config['log_level'],
            serialize=True
        )
        logger.add(
            lambda msg: print(msg, end=""),
            level=self.config['log_level']
        )
    
    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=4, max=10)
    )
    def call_llm(self, prompt):
        """Call LLM with retry logic and monitoring"""
        start_time = time.time()
        self.metrics['requests'] += 1
        
        try:
            logger.info("Making LLM request", extra={"prompt_length": len(prompt)})
            response = self.client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                timeout=self.config['timeout']
            )
            
            response_time = time.time() - start_time
            self.update_metrics(response_time)
            
            logger.info("LLM request successful", 
                       extra={"response_time": response_time, "tokens": response.usage.total_tokens})
            
            return response.choices[0].message.content
            
        except Exception as e:
            self.metrics['errors'] += 1
            logger.error("LLM request failed", extra={"error": str(e)})
            raise
    
    def update_metrics(self, response_time):
        """Update performance metrics"""
        current_avg = self.metrics['avg_response_time']
        total_requests = self.metrics['requests']
        self.metrics['avg_response_time'] = (current_avg * (total_requests - 1) + response_time) / total_requests
    
    def human_approval(self, action):
        """Get human approval for critical actions"""
        logger.warning("Human approval required", extra={"action": action})
        print(f"Action requires approval: {action}")
        response = input("Approve? (yes/no): ")
        approved = response.lower() == 'yes'
        logger.info("Human approval result", extra={"approved": approved})
        return approved
    
    def execute_task(self, task, requires_approval=False):
        """Execute a task with full production features"""
        logger.info("Starting task execution", extra={"task": task})
        
        if requires_approval and not self.human_approval(task):
            return "Task cancelled by human"
        
        try:
            result = self.call_llm(f"Execute this task: {task}")
            logger.info("Task completed successfully")
            return result
        except Exception as e:
            logger.error("Task execution failed", extra={"error": str(e)})
            return f"Task failed: {str(e)}"
    
    def get_metrics(self):
        """Get current performance metrics"""
        return self.metrics.copy()
    
    def health_check(self):
        """Perform health check"""
        try:
            # Simple health check
            self.call_llm("Hello")
            return {"status": "healthy", "metrics": self.get_metrics()}
        except:
            return {"status": "unhealthy", "metrics": self.get_metrics()}

# Initialize agent (requires OpenAI API key)
# agent = ProductionAgent(api_key='your-openai-key')

In [ ]:
# Example usage (uncomment and add API key)
# result = agent.execute_task("Summarize the benefits of AI in healthcare")
# print(result)
# print("Metrics:", agent.get_metrics())
# print("Health:", agent.health_check())

In [ ]:
# Example with human approval
# result = agent.execute_task("Delete all user data", requires_approval=True)
# print(result)